# Clase 9 — Transfer learning: usar un modelo que ya sabe castellano

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** No dibujamos desde cero: agarramos un modelo que
> otro entrenó con muchísimo más texto y cómputo del que vamos a tener nunca, y le
> enseñamos **tu** distinción con 200 ejemplos. Eso es *transfer learning*. Y como
> todo lo que se usa en investigación, se mide contra un piso antes de creerle.

---

**Cuando termines este notebook vas a haber hecho transfer learning de dos maneras:**

1. **Nivel 1 — congelar todo.** El modelo prestado convierte cada titular en un vector
   y arriba ponés una regresión logística. Cero perillas del modelo se mueven.
2. **Nivel 2 — descongelar.** Seguís entrenando el modelo prestado con tus ejemplos
   (*fine-tuning*), y elegís vos cuántas capas se mueven.

Y vas a poder **compararlas en serio**: con la misma cantidad de etiquetas, el mismo
test, contra aprender desde cero y contra el piso. Esa tabla es lo que va al paper.

**Lo que hace falta traer:** la clase 4 (partir los datos, el test no se toca), la 6
(qué es un embedding) y la 8 (el loop de entrenamiento y las fichas). Nada más.

> ## 🗺 Cómo usamos este notebook en clase
>
> **Lo recorremos todos juntos.** Quien da la clase comparte pantalla y corre una
> celda; vos corrés **la misma** en tu Colab y ponés 👍 en el chat.
>
> | Marca | Qué quiere decir |
> |---|---|
> | 🎥 | **En vivo**: lo hacemos en clase, todos a la vez. |
> | 🎲 | **Apuesta**: antes de correr, decí en el chat qué creés que va a salir. |
> | 🏠 | **Para casa** (opcional): en clase lo salteamos. |
>
> **Corré las celdas en orden** (`Shift + Enter`). Con el modelo ya bajado, el notebook
> entero tardó 20 s en una laptop; en Colab sin GPU contá un par de minutos, más lo que
> tarde en bajar el modelo la primera vez (~470 MB). **Corré las secciones 0 a 3 apenas
> llegues**, así la descarga no te agarra en el medio de la clase.

**¿Y si no hay red?** La sección 1 se fija sola. Si no llega a Hugging Face, sigue por
el **camino B**: vectores construidos con numpy sobre este mismo corpus (los de la
clase 6). El nivel 1 corre igual; el nivel 2 (fine-tuning) necesita el modelo, así que
en el camino B te muestra los números que medimos nosotros.

**El material extra** está en la carpeta `extra/`, al lado de este notebook. Empezá
por `extra/README.md`: ahí está la clase completa de dos horas (evaluación a fondo, el
umbral, la trampa del corpus, el acuerdo entre codificadores) y un notebook de
**recetas**, cada una una implementación corta y lista para copiar: `sentence-transformers`,
`Trainer` de Hugging Face, zero-shot, y cómo aplicarlo a un CSV tuyo.

## 🎥 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el repo y
deja todo a mano; en tu máquina no hace prácticamente nada. Corrémela y seguí de largo.

> **En Colab, opcional:** *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.
> El notebook lo detecta solo. No hace falta: en CPU ninguna celda tarda más que unos segundos.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "main"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports y semilla

Fijamos la semilla en 1983 **siempre**. Hoy importa más que de costumbre: el
fine-tuning con pocas etiquetas **cambia bastante de una semilla a otra**, y lo vas a
ver con tus propios ojos en la sección 6.

In [ ]:
import re
import time

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()

---

## 🎥 1. ¿Tenemos Hugging Face o no?

Hugging Face es el lugar donde la gente publica modelos pre-entrenados: un repositorio
de "informantes" que podés bajar. Antes de escribir una línea de análisis, averiguamos
si llegamos. La celda **no rompe nunca**: prueba, y si no puede, lo dice y sigue.

Vamos a usar **un solo modelo**, `paraphrase-multilingual-MiniLM-L12-v2`: chico (para
lo que se usa hoy), multilingüe —anda en castellano— y entrenado justamente para que
oraciones que dicen lo mismo queden cerca. Lo fijamos con su **revisión exacta** (el hash
del commit en el Hub): citar un modelo sin versión es como citar un libro sin edición.

In [ ]:
FORZAR_CAMINO_B = False     # True = no intentes bajar nada (el modelo pesa ~470 MB)

MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
REVISION = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"   # verificada el 2026-11-13

HAY_HF = False
MOTIVO_SIN_HF = ""

if FORZAR_CAMINO_B:
    MOTIVO_SIN_HF = "lo pediste con FORZAR_CAMINO_B = True"
else:
    try:
        import torch
        import transformers  # noqa: F401
        import urllib.request
        urllib.request.urlopen("https://huggingface.co", timeout=8)
        HAY_HF = True
        transformers.logging.set_verbosity_error()      # menos ruido en pantalla
        transformers.utils.logging.disable_progress_bar()
    except ImportError as e:
        MOTIVO_SIN_HF = f"falta una biblioteca ({e.name})"
    except Exception as e:
        MOTIVO_SIN_HF = f"no llego a huggingface.co ({type(e).__name__})"

if HAY_HF:
    DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"CAMINO A: Hugging Face disponible. Vamos a bajar {MODELO}.")
    print(f"   Corremos en: {DISPOSITIVO}")
else:
    print("CAMINO B (sin red): seguimos con los vectores propios del curso.")
    print(f"   Motivo: {MOTIVO_SIN_HF}.")
    print("   El nivel 1 corre igual. El nivel 2 (fine-tuning) te muestra lo que medimos.")

---

## 🎥 2. Los datos, la tarea y **el piso**

El corpus de prensa del curso: **1471 notas reales en castellano** (*Spanish Fake News
Corpus 2.0*, CC BY 4.0; procedencia en `datos/README.md` §1.3). La tarea es una que hacés
todo el tiempo aunque no la llames así: **armar un subcorpus temático**. En concreto:
*¿esta nota es sobre política?*

Tres cosas, antes de tocar un modelo:

1. **El piso**: cuánto acierta un clasificador que no mira nada y dice siempre «no».
2. **La partición** (clase 4): un 25 % de test que **no se toca** hasta el final.
3. **Cuántas etiquetas tenés**. En la vida real nadie tiene 1103 titulares
   etiquetados: tiene los que pudo codificar. Por eso entrenamos con una muestra de
   `N_ETIQUETAS` (200 = unas dos horas de codificación a mano), **la misma para todas
   las estrategias**. Sin eso, la comparación es trampa.

In [ ]:
notas = datos.titulares_prensa()
titulares = [n["titular"] for n in notas]
y = np.array([int(n["tema"] == "Politica") for n in notas])   # 1 = es sobre política

# La partición de la clase 4: estratificada, y el test no se toca.
i_train, i_test = train_test_split(np.arange(len(y)), test_size=0.25,
                                   stratify=y, random_state=1983)

prev = y[i_test].mean()
print(f"{len(notas)} titulares. De cada 100, {prev * 100:.0f} son sobre política.")
print(f"EL PISO: decir siempre «no» acierta el {(1 - prev) * 100:.0f} % del test "
      f"y encuentra CERO notas de política (F1 = 0.00).")
print(f"\nDisponibles para entrenar: {len(i_train)}.  Test, guardado bajo llave: {len(i_test)}.")


def muestra_de_entrenamiento(n, semilla=1983):
    # n titulares de train, al azar, con la misma proporción de positivos.
    rng = np.random.default_rng(semilla)
    pos = [i for i in i_train if y[i] == 1]
    neg = [i for i in i_train if y[i] == 0]
    k = max(2, int(round(n * len(pos) / len(i_train))))
    return np.concatenate([rng.choice(pos, k, replace=False),
                           rng.choice(neg, n - k, replace=False)])


N_ETIQUETAS = 200          # 🏠 probá 50, y después len(i_train) (todas)
idx = muestra_de_entrenamiento(N_ETIQUETAS)
print(f"\nHoy entrenamos con {len(idx)} titulares etiquetados "
      f"({y[idx].sum()} de política). Lo mismo para todas las estrategias.")

### La función `reportar()`

**Regla de la casa: prohibido reportar una métrica sin su frase en castellano.** Si no
podés decir *"de cada 100 notas que el modelo llamó política, N lo eran"*, no entendiste
tu propio resultado. Esta función imprime las frases y el piso al lado. Copiala.

In [ ]:
def reportar(pred, nombre, dibujar=False):
    # Las frases, el piso y (si querés) la matriz. Siempre contra el MISMO test.
    print(f"--- {nombre} ---")
    g.frase_de_metricas(y[i_test], pred, fenomeno="nota sobre política")
    if dibujar:
        g.matriz_de_confusion(y[i_test], pred,
                              etiquetas=["no es política", "es política"],
                              nombre_positivo="notas sobre política",
                              titulo=f"Matriz de confusión — {nombre}")
    return g.metricas_binarias(y[i_test], pred)


resultados = {}            # acá vamos juntando la tabla de la sección 7

---

## 🎥 3. El informante: un modelo que ya sabe castellano

Bajamos el modelo. Son dos piezas, las mismas de la clase 8:

- el **tokenizador**, que parte el texto en fichas;
- el **modelo**, que convierte cada ficha en un vector, mirando a las demás (12 bloques
  transformer como los de la clase 7, apilados).

Nadie le enseñó qué es "una nota sobre política". Le enseñaron castellano —y
unos cincuenta idiomas más— con muchísimo texto. **Esa es la parte que tomamos prestada.**

In [ ]:
if HAY_HF:
    from transformers import AutoModel, AutoTokenizer
    try:
        tok = AutoTokenizer.from_pretrained(MODELO, revision=REVISION)
        base = AutoModel.from_pretrained(MODELO, revision=REVISION).to(DISPOSITIVO).eval()
    except Exception as e:
        # intentar -> si falla, avisar y seguir por el plan B
        print(f"[!] No pude bajar {MODELO} ({type(e).__name__}). Sigo por el camino B.")
        HAY_HF = False

if HAY_HF:
    total = utils.contar_parametros(base, solo_entrenables=False)
    vocab = base.embeddings.word_embeddings.weight.numel()
    print(f"Perillas del modelo prestado: {utils.formatear_cantidad(total)}")
    print(f"   la tabla de fichas (una fila por cada una de las "
          f"{base.config.vocab_size:,} fichas): {utils.formatear_cantidad(vocab)}")
    print(f"   los {base.config.num_hidden_layers} bloques transformer: "
          f"{utils.formatear_cantidad(total - vocab)}")
    print("Tu mini-GPT de la clase 8 tenía ~1,5 M. Este es unas 80 veces más grande.")

    ejemplo = titulares[3]
    print(f"\nUn titular:  {ejemplo}")
    print(f"Sus fichas:  {tok.tokenize(ejemplo)}")
else:
    print("Camino B: no hay modelo para inspeccionar. Seguí: la sección 4 corre igual.")

---

## 🎲 Apuesta, antes de seguir

Vamos a entrenar dos clasificadores con **los mismos 200 titulares**:

- **Nivel 1:** el modelo prestado lee cada titular y lo convierte en un vector de 384
  números; arriba, una regresión logística con **385 perillas**.
- **Desde cero:** una columna por cada palabra y par de palabras de los 200 titulares
  (TF-IDF), y una regresión logística que aprende el peso de cada una con **miles de
  perillas** y ningún conocimiento previo.

**¿Cuál saca mejor F1 en el test?** Escribí en el chat *prestado* o *desde cero*, y
por cuánto.

---

## 🎥 4. Nivel 1 — congelar todo: el modelo lee, vos decidís

Pasamos cada titular por el modelo **sin entrenarlo** (`torch.no_grad()`: ni una
perilla se mueve) y nos quedamos con un vector por titular: el promedio de los vectores
de sus fichas. Eso es un *sentence embedding*. Es el mapa de la clase 6, pero el mapa lo
hizo otro, con muchísimo más texto.

In [ ]:
if HAY_HF:
    def vectorizar(textos, lote=64):
        # Cada texto -> un vector. El modelo está CONGELADO: no_grad, eval.
        salida = []
        for k in range(0, len(textos), lote):
            entrada = tok(list(textos[k:k + lote]), padding=True, truncation=True,
                          max_length=64, return_tensors="pt").to(DISPOSITIVO)
            with torch.no_grad():
                h = base(**entrada).last_hidden_state             # un vector por ficha
            m = entrada["attention_mask"].unsqueeze(-1).float()
            v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)          # promedio de fichas
            salida.append(v.cpu().numpy())
        M = np.vstack(salida)
        return M / np.linalg.norm(M, axis=1, keepdims=True)

    COMO_VECTORIZAMOS = f"{MODELO} (revisión {REVISION[:7]}), congelado"
else:
    # Camino B: los embeddings de la clase 6 (coocurrencia -> PPMI -> SVD), con numpy.
    # Se construyen con todos los textos sin mirar etiquetas: no es fuga (clase 4).
    corpus_entero = " \n ".join(n["titular"] + " " + n["texto"]
                                for n in datos.titulares_prensa(con_texto=True))
    V, palabras = datos.embeddings_ppmi_svd(corpus_entero, dim=100, top=2000,
                                            ventana=5, minimo=5)
    _indice = {p: i for i, p in enumerate(palabras)}
    _tok = datos.TokenizadorPalabras(minusculas=True)

    def vectorizar(textos):
        M = np.zeros((len(textos), V.shape[1]), dtype=np.float32)
        for k, t in enumerate(textos):
            ws = [w for w in _tok.partir(t.lower()) if w in _indice]
            if ws:
                M[k] = V[[_indice[w] for w in ws]].mean(axis=0)
        return M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-9)

    COMO_VECTORIZAMOS = "PPMI+SVD propios sobre este corpus (camino B)"

with utils.medir_tiempo("vectorizar los 1471 titulares"):
    X = vectorizar(titulares)
print(f"Cada titular es ahora un punto en {X.shape[1]} dimensiones.")

nivel1 = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[idx], y[idx])
pred_nivel1 = nivel1.predict(X[i_test])
resultados["nivel 1 · congelado + logística"] = reportar(
    pred_nivel1, "nivel 1 · congelado + logística", dibujar=True)

---

## 🎥 5. La otra punta: nada prestado

Ahora **sin informante**. TF-IDF: una columna por palabra (o par de palabras) que
aparece en tus 200 titulares, y la regresión aprende el peso de cada una. Todo lo que
sabe sale de tus ejemplos. Muchas más perillas, ningún conocimiento previo: la clase 4
ya te dijo qué esperar.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
A_train = tfidf.fit_transform([titulares[i] for i in idx])
A_test = tfidf.transform([titulares[i] for i in i_test])
print(f"Desde cero: {A_train.shape[1]:,} perillas que aprender con {len(idx)} ejemplos "
      f"(el nivel 1 tenía {X.shape[1] + 1}).\n")

desde_cero = LogisticRegression(max_iter=2000, class_weight="balanced").fit(A_train, y[idx])
pred_cero = desde_cero.predict(A_test)
resultados["desde cero · TF-IDF + logística"] = reportar(
    pred_cero, "desde cero · TF-IDF + logística")

f1_1 = resultados["nivel 1 · congelado + logística"]["f1"]
f1_0 = resultados["desde cero · TF-IDF + logística"]["f1"]
print(f"\nCon {len(idx)} etiquetas: prestado {f1_1:.2f} contra desde cero {f1_0:.2f}. "
      f"¿Acertaste la apuesta?")

---

## 🎥 6. Nivel 2 — descongelar: *fine-tuning*

En el nivel 1 el modelo leía y no aprendía nada. Ahora lo dejamos **seguir
aprendiendo** con tus 200 ejemplos. Armamos un `nn.Module` (clase 5 y 8) con dos
pisos:

```
titular → [ modelo prestado: 12 bloques ] → promedio de fichas → [ nn.Linear(384, 2) ] → sí / no
              ↑ prestado                                              ↑ nuevo: tu distinción
```

y elegís **cuántos de los 12 bloques se mueven** con una sola perilla, `CAPAS_LIBRES`:

| `CAPAS_LIBRES` | qué se entrena | perillas que se mueven |
|---|---|---|
| `0` | sólo la capa nueva de arriba | 770: es casi el nivel 1 otra vez |
| `2` | la capa nueva + los 2 bloques de más arriba | ~3,5 M |
| `12` | la capa nueva + los 12 bloques | ~21 M |

(La tabla de fichas —96 M de las 118 M perillas— queda congelada siempre: es lo más
genérico que tiene el modelo, y no hay 200 titulares que la mejoren.)

Es la jerarquía de la clase 5: abajo lo genérico, arriba lo específico. **Congelar es
bajar la capacidad a propósito** para que 200 ejemplos alcancen.

In [ ]:
if HAY_HF:
    import torch.nn as nn
    import torch.nn.functional as F

    class Clasificador(nn.Module):
        # El modelo prestado + una capa nueva arriba para TU distinción.
        def __init__(self, capas_libres):
            super().__init__()
            self.base = AutoModel.from_pretrained(MODELO, revision=REVISION)
            self.cabeza = nn.Linear(self.base.config.hidden_size, 2)     # lo nuevo
            for p in self.base.parameters():                             # congelar todo...
                p.requires_grad = False
            if capas_libres > 0:                                         # ...y soltar arriba
                for bloque in self.base.encoder.layer[-capas_libres:]:
                    for p in bloque.parameters():
                        p.requires_grad = True

        def forward(self, entrada):
            h = self.base(**entrada).last_hidden_state
            m = entrada["attention_mask"].unsqueeze(-1).float()
            return self.cabeza((h * m).sum(1) / m.sum(1).clamp(min=1e-9))

    for capas in [0, 2, 12]:
        modelo = Clasificador(capas)
        print(f"CAPAS_LIBRES = {capas:>2}: se mueven "
              f"{utils.formatear_cantidad(utils.contar_parametros(modelo)):>7} "
              f"de {utils.formatear_cantidad(utils.contar_parametros(modelo, solo_entrenables=False))}")
    del modelo
else:
    print("Camino B: el fine-tuning necesita el modelo pre-entrenado. Leé la celda que sigue.")

### El entrenamiento: las cinco estaciones de la clase 4, sin una idea nueva

Adentro del `Trainer` de Hugging Face (que vas a ver en el material extra) no pasa nada
distinto de esto. Lo único que cambia respecto de la clase 8 es **de dónde arrancan las
perillas**: no de números al azar, sino de lo que ya sabía el modelo.

Dos detalles que importan:
- **Learning rate chico** (`5e-5`) cuando se mueven los bloques prestados: querés
  *ajustarlos*, no borrar lo que saben. Si sólo entrenás la capa nueva, uno más grande.
- **Pocas épocas** (3). Con 200 ejemplos, más vueltas es calcar (clase 4).

In [ ]:
CAPAS_LIBRES = 12      # 🎲 TODO 1: probá 0 y 2
SEMILLA = 1983         # 🎲 TODO 1: probá 1984 y 1985


def afinar(idx_train, capas_libres=CAPAS_LIBRES, semilla=SEMILLA, epocas=3):
    # Fine-tuning a mano. Devuelve las predicciones sobre el test.
    torch.manual_seed(semilla)
    rng = np.random.default_rng(semilla)
    modelo = Clasificador(capas_libres).to(DISPOSITIVO)
    lr = 1e-3 if capas_libres == 0 else 5e-5
    optimizador = torch.optim.AdamW(
        [p for p in modelo.parameters() if p.requires_grad], lr=lr)
    prev_tr = y[i_train].mean()
    peso = torch.tensor([1.0, (1 - prev_tr) / prev_tr], dtype=torch.float,
                        device=DISPOSITIVO)                    # como class_weight="balanced"

    for epoca in range(epocas):
        modelo.train()
        for lote in np.array_split(rng.permutation(idx_train), max(1, len(idx_train) // 16)):
            entrada = tok([titulares[i] for i in lote], padding=True, truncation=True,
                          max_length=64, return_tensors="pt").to(DISPOSITIVO)
            objetivo = torch.tensor(y[lote], device=DISPOSITIVO)
            puntajes = modelo(entrada)                                  # 1. PREDECIR
            perdida = F.cross_entropy(puntajes, objetivo, weight=peso)  # 2. COMPARAR
            optimizador.zero_grad()                                     # 3. BORRAR
            perdida.backward()                                          # 4. ANOTAR
            optimizador.step()                                          # 5. MOVER
        print(f"   época {epoca + 1}: loss del último lote {perdida.item():.3f}")

    modelo.eval()
    with torch.no_grad():
        pred = [modelo(tok([titulares[i] for i in i_test[k:k + 64]], padding=True,
                           truncation=True, max_length=64,
                           return_tensors="pt").to(DISPOSITIVO)).argmax(-1).cpu().numpy()
                for k in range(0, len(i_test), 64)]
    return np.concatenate(pred), modelo


if HAY_HF:
    with utils.medir_tiempo(f"fine-tuning con {len(idx)} etiquetas, {CAPAS_LIBRES} capas libres"):
        pred_nivel2, modelo_afinado = afinar(idx)
    nombre_n2 = f"nivel 2 · fine-tuning, {CAPAS_LIBRES} capas libres"
    resultados[nombre_n2] = reportar(pred_nivel2, nombre_n2)
else:
    print("Camino B. Lo que medimos con el camino A (los mismos 200 titulares, mismo")
    print("test, SEMILLA = 1983 / 1984 / 1985; medido el 2026-11-13):")
    print("   CAPAS_LIBRES =  0  ->  F1 0.61 / 0.55 / 0.58")
    print("   CAPAS_LIBRES =  2  ->  F1 0.61 / 0.61 / 0.62")
    print("   CAPAS_LIBRES = 12  ->  F1 0.60 / 0.57 / 0.65")
    print("El nivel 1 (congelado + logística) dio 0.64; aprender desde cero, 0.44.")

### 🎲 `TODO 1` — mové las dos perillas

Antes de correr, apostá en el chat: **¿descongelar más da siempre mejor F1?**

1. Cambiá `CAPAS_LIBRES` a `0` y volvé a correr la celda de arriba
   (es una sola: define `afinar` y entrena). Después a `2`.
2. Volvé a `12` y cambiá `SEMILLA` a `1984` y a `1985`.

Anotá los F1 en la tabla de abajo. **¿Cuánto se mueve el resultado sólo por cambiar la
semilla?** Y entonces: ¿la diferencia entre `0` y `12` es más grande o más chica que
eso? Si es más chica, no podés afirmar que una le gana a la otra.

| `CAPAS_LIBRES` | semilla 1983 | 1984 | 1985 |
|---|---|---|---|
| 0  | | | |
| 2  | | | |
| 12 | | | |

---

## 🎥 7. La tabla honesta

Todo junto, **con el mismo test y la misma cantidad de etiquetas**. Le sumamos las dos
filas que no necesitan entrenar nada: el piso y un diccionario de palabras clave escrito
a mano en cinco minutos (lo que hace muchísima investigación publicada). **Esta tabla
es la que va al paper.**

In [ ]:
DICCIONARIO = (r"gobierno|president|diputad|senad|partido|elecci|"
               r"ministr|congreso|pol[ií]tic|candidat|campa[ñn]a electoral")
pred_dicc = np.array([int(bool(re.search(DICCIONARIO, titulares[i], re.I))) for i in i_test])

prev_test = float(y[i_test].mean())
filas = [("el piso: decir siempre «no»", 1 - prev_test, 0.0, 0.0, 0.0, "0"),
         ("diccionario de palabras clave",
          *[g.metricas_binarias(y[i_test], pred_dicc)[k]
            for k in ("accuracy", "precision", "recall", "f1")], "0")]
filas += [(nombre, m["accuracy"], m["precision"], m["recall"], m["f1"], str(len(idx)))
          for nombre, m in resultados.items()]

print(f"{'estrategia':<40}{'acc':>6}{'prec':>6}{'rec':>6}{'F1':>6}   etiquetas")
print("-" * 82)
for nombre, acc, prec, rec, f1, etiquetas in filas:
    print(f"{nombre:<40}{acc:>6.2f}{prec:>6.2f}{rec:>6.2f}{f1:>6.2f}   {etiquetas}")
print("-" * 82)
print(f"Mismo test para todas: {len(i_test)} titulares que ningún modelo vio.")
print("Fijate en la accuracy del piso: alta, y no encontró ni una nota de política.")

### `TODO 2` — de la tabla a una decisión

Escribí la recomendación de **dos oraciones** que le darías a un colega que tiene 200
titulares etiquetados y quiere clasificar 50.000. Tiene que decir **qué estrategia** y
**por qué**, con un número de la tabla. Y si recomendás el fine-tuning, decí cuánto se
movía con la semilla.

*Pista.* Cuando dos opciones empatan, la más barata, más estable y más fácil de
explicarle a un revisor gana el empate.

🏠 **Para casa:** cambiá `N_ETIQUETAS` a `50`, y después a `len(i_train)` (todas, 1103),
y corré todo de nuevo. ¿Con cuántas etiquetas "desde cero" alcanza a lo prestado? La
respuesta completa, con cinco semillas por tamaño, está en `viz/zero-shot-vs-finetune.html`.

**Tu respuesta:** _(escribí acá)_

---

## 🎥 8. Para llevarte: `clasificar_corpus()`

Esto es lo concreto. Una función lista para que le pases **tus** textos, con el
clasificador del nivel 1 (el barato, el estable). Devuelve el **puntaje**, no sólo el
sí/no: sin el puntaje no podés mover el umbral (la viz de la matriz de confusión).

In [ ]:
def clasificar_corpus(textos, modelo=nivel1, umbral=0.5):
    # Devuelve [{texto, puntaje, prediccion}]. El puntaje NO es una probabilidad calibrada.
    puntajes = modelo.predict_proba(vectorizar(list(textos)))[:, 1]
    return [{"texto": t, "puntaje": float(p), "prediccion": int(p >= umbral)}
            for t, p in zip(textos, puntajes)]


# Titulares argentinos: otro país y otra época que los del entrenamiento.
for r in clasificar_corpus(["El oficialismo y la oposición se cruzaron por el presupuesto",
                            "Paro de colectivos en el AMBA por reclamo salarial",
                            "Récord de temperatura en el sur: 41 grados en pleno enero"]):
    print(f"  [{'SÍ' if r['prediccion'] else 'no'}  {r['puntaje']:.2f}]  {r['texto']}")
print("\nFuera del dominio de entrenamiento (prensa mexicana y española de 2018-2021)")
print("esto NO está validado. Se revalida con etiquetas propias antes de publicar nada.")

print("\nLa ficha del modelo, que va SÍ O SÍ en tu sección de métodos:")
print(f"   vectorización : {COMO_VECTORIZAMOS}")
print(f"   clasificador  : regresión logística (class_weight='balanced'), scikit-learn")
print(f"   entrenamiento : {len(idx)} titulares etiquetados, muestra estratificada, semilla 1983")
print(f"   evaluación    : {len(i_test)} titulares de test, nunca usados para elegir nada")
print(f"   F1 en test    : {resultados['nivel 1 · congelado + logística']['f1']:.2f}  "
      f"(piso: 0.00; diccionario: {filas[1][4]:.2f})")

---

## Qué te llevás

1. **Transfer learning es pedir prestado lo genérico y entrenar sólo lo tuyo.** Con
   pocas etiquetas, lo prestado le gana por lejos a aprender desde cero. Lo mediste.
2. **Congelar o descongelar es elegir cuántas perillas movés**, y es la misma decisión
   de la clase 4: más perillas piden más datos. Descongelar no gana siempre, y con pocas
   etiquetas el resultado baila con la semilla. **Probá primero el nivel 1**; si vas a
   hacer fine-tuning, medilo contra el nivel 1 con varias semillas.
3. **Nada de esto se cree sin la tabla**: mismo test, mismas etiquetas, el piso y el
   diccionario al lado, y cada métrica con su frase.

**En tu investigación (proyecto de mediano plazo, no tarea para la clase 10):**

1. Definí la categoría por escrito, con los casos límite resueltos.
2. Etiquetá **300 al azar, a ciegas**: 200 para entrenar y 100 para evaluar (mejor si
   después sumás más test). Son unas 2,5 horas.
3. Corré este notebook cambiando `titulares` e `y` por los tuyos (la receta 4 del
   material extra lo hace desde un CSV).
4. Reportá la tabla de la sección 7, la ficha de la sección 8 y 30 errores leídos.

**Lo que queda prometido.** Tenemos un número bueno. ¿Prueba que medimos lo que decimos
que medimos? No. La clase 10 es sobre esa distancia.

---

### Material extra (`extra/README.md`)

| archivo | qué tiene | cuánto lleva |
|---|---|---|
| `extra/recetas-transfer-learning.ipynb` | **implementaciones cortas, una por receta**: `sentence-transformers` en 3 líneas, fine-tuning con `Trainer`, zero-shot, tu propio CSV | 1 hora |
| `extra/notebook-completo.ipynb` | la clase completa de dos horas: búsqueda semántica, la curva de etiquetas, el umbral, leer los errores, la trampa del corpus, el acuerdo entre codificadores | 2 horas |
| `viz/` | las tres visualizaciones de la clase, sin internet | 15 min |
| `referencias.md` | qué leer y en qué orden; empezá por Barberá et al. (2021) | — |